In [24]:
from langchain.vectorstores import Chroma
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.storage import InMemoryStore
from langchain.document_loaders import TextLoader
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo-1106", temperature=0)
from langchain.retrievers import ParentDocumentRetriever
from langchain.text_splitter import MarkdownHeaderTextSplitter

In [25]:
import os
import openai

In [26]:
# english to english tamil convertor

In [27]:
llm.predict("hello world")

'Hello! How can I assist you today?'

In [28]:
# Open the file in read mode ('r')
with open('./docs/sop.md', 'r') as file:
    # Read the contents of the file
    file_contents = file.read()

In [29]:
markdown_document = file_contents

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
md_header_splits = markdown_splitter.split_text(markdown_document)

In [30]:
md_header_splits

[Document(page_content="This extensive monthly report provides a deep dive into ABC Corp's Treasury and Market Operations. It covers detailed financial activities, market positions, risk management strategies, and actionable insights. This report is a critical tool for guiding strategic decisions and assessing our financial health.", metadata={'Header 1': 'ABC Corp Treasury and Market Operations Report', 'Header 2': 'Executive Summary'}),
 Document(page_content='### Cash Management\n- **Current Cash Position**: Detailed report on our cash reserves, highlighting $XX million.\n- **Cash Flow Projections**: Month-by-month cash flow projections for the next quarter, with analysis of trends and variances.\n- **Liquidity Analysis**: Comprehensive liquidity analysis, assessing current assets, upcoming liabilities, and strategies for maintaining optimal liquidity.\n- **Banking Relationships and Negotiations**: Evaluation of current banking relationships, negotiation strategies for better rates,

In [38]:
# This text splitter is used to create the parent documents
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)
# This text splitter is used to create the child documents
# It should create documents smaller than the parent
child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)
# The vectorstore to use to index the child chunks
vectorstore = Chroma(
    collection_name="split_parents", embedding_function=OpenAIEmbeddings()
)
# The storage layer for the parent documents
store = InMemoryStore()

In [39]:
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    search_kwargs={'k':2}
)

In [40]:
retriever.add_documents(md_header_splits)

In [41]:
retriever.get_relevant_documents("where to get the best pasta?")

[Document(page_content='### Cash Management\n- **Current Cash Position**: Detailed report on our cash reserves, highlighting $XX million.\n- **Cash Flow Projections**: Month-by-month cash flow projections for the next quarter, with analysis of trends and variances.\n- **Liquidity Analysis**: Comprehensive liquidity analysis, assessing current assets, upcoming liabilities, and strategies for maintaining optimal liquidity.\n- **Banking Relationships and Negotiations**: Evaluation of current banking relationships, negotiation strategies for better rates, and diversification of banking partners.\n- You can get the best pasta at the south union at kembangan  \n### Investment Portfolio\n- **Portfolio Composition**: Detailed analysis of our portfolio, including asset allocation, sector weighting, and maturity profiles.\n- **Performance Analysis**: In-depth performance review against various benchmarks and historical performance, with a focus on return on investment.\n- **Strategic Adjustments

In [ ]:
vectorstore.similarity_search("where to get the best pasta?")

In [42]:
search = RetrievalQA.from_chain_type(
    llm=llm, chain_type="stuff", retriever=retriever
)

In [43]:
answer = search("where to get the best pasta?")
print(answer)

{'query': 'where to get the best pasta?', 'result': "I don't have that information."}
